<a href="https://colab.research.google.com/github/Ebasurtos/AHPC/blob/main/LAb2v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

# ======================
# 2. DATASETS & TRANSFORMS
# ======================
class SegmentationTransform:
    def __init__(self, size):
        self.img_transform = T.Compose([
            T.Resize(size),
            T.ToTensor()
        ])
        self.mask_transform = T.Compose([
            T.Resize(size, interpolation=T.InterpolationMode.NEAREST),
            T.PILToTensor(),
            T.Lambda(lambda x: x.squeeze(0).long())
        ])

    def __call__(self, img, mask):
        return self.img_transform(img), self.mask_transform(mask)

class RemapLabels:
    def __init__(self, mode='voc'):
        self.mode = mode

    def __call__(self, mask):
        if self.mode == 'pet':
            return mask - 1 # 1,2,3 -> 0,1,2
        elif self.mode == 'voc':
            remapped = torch.zeros_like(mask)
            remapped[mask == 15] = 1 # Person
            remapped[(mask > 0) & (mask != 15) & (mask != 255)] = 2 # Other
            return remapped
        return mask

# Initialization
size = (128, 128)
trans_voc = SegmentationTransform(size)
trans_pet = SegmentationTransform(size)
trans_fake = T.Compose([T.Resize(size), T.ToTensor()])

voc_raw = torchvision.datasets.VOCSegmentation(root='./data', year='2012', image_set='train', download=True, transforms=trans_voc)
pet_raw = torchvision.datasets.OxfordIIITPet(root='./data', split='trainval', download=True, target_types='segmentation', transforms=trans_pet)
fake = torchvision.datasets.FakeData(size=200, image_size=(3,128,128), num_classes=3, transform=trans_fake)

class RemappedDataset(torch.utils.data.Dataset):
    def __init__(self, base, mode):
        self.base = base
        self.remap = RemapLabels(mode)
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        img, mask = self.base[idx]
        return img, self.remap(mask)

loaders = {
    "VOC": DataLoader(RemappedDataset(voc_raw, 'voc'), batch_size=4, shuffle=True),
    "PET": DataLoader(RemappedDataset(pet_raw, 'pet'), batch_size=4, shuffle=True),
    "FAKE": DataLoader(fake, batch_size=4, shuffle=True)
}

# ======================
# 3. MODELO ECENET
# ======================
class FeatureReconstruction(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c//2, 1)
        self.conv2 = nn.Conv2d(c//2, c//2, 3, padding=1)
        self.out = nn.Conv2d(c, c, 1)
    def forward(self,x):
        a = self.conv1(x)
        b = self.conv2(a)
        return self.out(torch.cat([a,b],1))

class ECE(nn.Module):
    def __init__(self,c,nc):
        super().__init__()
        self.conv = nn.Conv2d(c,nc,1)
        self.fc = nn.Linear(nc,c)
    def forward(self,x):
        m = self.conv(x)
        p = F.adaptive_avg_pool2d(m,(1,1)).view(x.size(0),-1)
        e = self.fc(p)
        return e,m

class SAU(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.attn = nn.MultiheadAttention(c,4,batch_first=True)
    def forward(self,f,e):
        B,C,H,W = f.shape
        f = f.view(B,C,-1).permute(0,2,1)
        e = e.unsqueeze(1)
        out,_ = self.attn(f,e,e)
        return out.permute(0,2,1).view(B,C,H,W)

class ECENet(nn.Module):
    def __init__(self,nc=3):
        super().__init__()
        base = torchvision.models.resnet18(weights="DEFAULT")
        self.backbone = nn.Sequential(*list(base.children())[:-2])
        self.fr = FeatureReconstruction(512)
        self.ece = ECE(512,nc)
        self.sau = SAU(512)
        self.dec = nn.Conv2d(512,nc,1)
    def forward(self,x):
        f = self.backbone(x)
        f = self.fr(f)
        e,m = self.ece(f)
        f = self.sau(f,e)
        out = self.dec(f)
        return F.interpolate(out,(128,128)), m

# ======================
# 4. MÉTRICAS
# ======================
def compute_mIoU(pred, target, num_classes=3):
    pred = torch.argmax(pred, 1)
    ious = []
    for cls in range(num_classes):
        inter = ((pred == cls) & (target == cls)).sum().item()
        union = ((pred == cls) | (target == cls)).sum().item()
        if union > 0: ious.append(inter / union)
    return np.mean(ious) if ious else 0

# ======================
# 5. ENTRENAMIENTO
# ======================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ECENet(3).to(device)
opt = torch.optim.Adam(model.parameters(), 1e-4)
loss_fn = nn.CrossEntropyLoss()

history = {}
for name, loader in loaders.items():
    print(f"\nEntrenando en {name}")
    losses, ious = [], []
    for epoch in range(2):
        model.train()
        total_loss = 0
        for x, y in tqdm(loader):
            x, y = x.to(device), y.to(device)
            p, _ = model(x)
            loss = loss_fn(p, y)
            opt.zero_grad(); loss.backward(); opt.step()
            total_loss += loss.item()

        model.eval()
        all_ious = []
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                p, _ = model(x)
                all_ious.append(compute_mIoU(p, y))

        avg_iou = np.mean(all_ious)
        losses.append(total_loss/len(loader))
        ious.append(avg_iou)
        print(f"Epoch {epoch}: Loss {losses[-1]:.3f} | mIoU {avg_iou:.3f}")
    history[name] = {"loss": losses, "miou": ious}

# ======================
# 6. RESULTADOS
# ======================
plt.figure()
for k, v in history.items():
    plt.plot(v['miou'], label=k)
plt.legend(); plt.title("mIoU Evolution"); plt.show()


Entrenando en VOC


100%|██████████| 366/366 [03:06<00:00,  1.97it/s]


Epoch 0: Loss 0.640 | mIoU 0.339


100%|██████████| 366/366 [03:03<00:00,  1.99it/s]


Epoch 1: Loss 0.603 | mIoU 0.312

Entrenando en PET


...

Epoch 0: Loss 0.892 | mIoU 0.252


100%|██████████| 920/920 [07:57<00:00,  1.93it/s]


Epoch 1: Loss 0.866 | mIoU 0.268

Entrenando en FAKE


  0%|          | 0/50 [00:00<?, ?it/s]


RuntimeError: only batches of spatial targets supported (3D tensors) but got targets of dimension: 1